# Indic-Canary Marathi ASR: Controlled LoRA Fine-Tuning & Evaluation
**Author:** AI Research Engineer Candidate  
**Target:** AI4Bharat / Bodhan AI Take-Home Coding Assignment (IIT Madras)  
**Foundation Model:** `bodhan-ai/indic-transcribe-core` (1.22B FastConformer Canary)  
**Adaptation Strategy:** LoRA ($r=16, lpha=32$) on Decoder Self & Cross-Attention (3.15M trainable parameters)  
**Target Corpus:** `ai4bharat/Kathbath` Marathi (Strict Speaker-Disjoint Partition: 363 Train, 100 Held-Out Val)  
**Hardware Platform:** NVIDIA Tesla T4 GPU (14.56 GB Dedicated GDDR6 VRAM)


In [ ]:
# 1. Environment Inspection & Dependency Setup
import os
import sys
import torch

print("=" * 65)
print("KAGGLE GPU & PLATFORM TELEMETRY")
print("=" * 65)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Accelerator :", torch.cuda.get_device_name(0))
    print("Dedicated VRAM  :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

# Cleanly install verified dependencies
!pip uninstall -y torchao
!pip install -q peft==0.19.1 soundfile torchaudio fsspec pyarrow jiwer kaggle


In [ ]:
# 2. Hugging Face Authentication for Gated Assets
from huggingface_hub import login, whoami

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    hf_token = input("Enter your Hugging Face Access Token: ").strip()

login(token=hf_token)
user = whoami()
print("Authenticated successfully as:", user["name"])


In [ ]:
# 3. Model Snapshot Download & Architecture Loading
from huggingface_hub import snapshot_download

print("Downloading model snapshot (model.safetensors + configs + tokenizer)...")
model_dir = snapshot_download(
    repo_id="bodhan-ai/indic-transcribe-core",
    token=hf_token,
    allow_patterns=["*.safetensors", "*.py", "*.json", "*.model"]
)
print("Snapshot cached at:", model_dir)

if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

from modeling_indic_canary import IndicCanaryForConditionalGeneration
from feature_extraction_indic_canary import IndicCanaryFeatureExtractor
from tokenization_indic_canary import IndicCanaryTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading IndicCanary in float32 onto {device}...")
base_model = IndicCanaryForConditionalGeneration.from_pretrained(model_dir, dtype=torch.float32)
base_model = base_model.to(device)

fe = IndicCanaryFeatureExtractor.from_pretrained(model_dir, device=device)
tokenizer = IndicCanaryTokenizer.from_pretrained(model_dir)
print("Base model, Feature Extractor, and Tokenizer loaded successfully!")


In [ ]:
# 4. Inject LoRA on Decoder Cross & Self-Attention (r=16, alpha=32)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r".*decoder.*(query_net|value_net)",
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model = model.to(device).train()

trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p = sum(p.numel() for p in model.parameters())
print(f"LoRA Trainable Parameters: {trainable_p:,} / {total_p:,} ({trainable_p/total_p*100:.2f}%)")
assert trainable_p == 3145728, f"Unexpected trainable params: {trainable_p}"
print("Adaptation architecture mathematically verified across all 24 decoder layers!")


In [ ]:
# 5. Dataset Ingestion & Preprocessing Verification
import glob
import json
import soundfile as sf
import torchaudio

# Print /kaggle/input contents
if os.path.exists("/kaggle/input"):
    print("Mounted files in /kaggle/input:", os.listdir("/kaggle/input"))

# Locate dataset directory dynamically
manifest_matches = glob.glob("/kaggle/input/**/train_manifest.jsonl", recursive=True) +                    glob.glob("data/**/train_manifest.jsonl", recursive=True) +                    glob.glob("/kaggle/working/**/train_manifest.jsonl", recursive=True)

dataset_root = None
if manifest_matches:
    dataset_root = os.path.dirname(manifest_matches[0])
else:
    print("Dataset not mounted under /kaggle/input. Triggering self-healing API download...")
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        download_dir = "/kaggle/working/data/kaggle_dataset"
        os.makedirs(download_dir, exist_ok=True)
        api.dataset_download_files("ac2sny/kathbath-marathi-level4", path=download_dir, unzip=True)
        submatches = glob.glob(os.path.join(download_dir, "**/train_manifest.jsonl"), recursive=True)
        if submatches:
            dataset_root = os.path.dirname(submatches[0])
        else:
            dataset_root = download_dir
        print("Dataset downloaded successfully to:", dataset_root)
    except Exception as e:
        print("Self-healing download exception:", e)

if not dataset_root or not os.path.exists(os.path.join(dataset_root, "train_manifest.jsonl")):
    raise FileNotFoundError(f"Could not locate dataset root. /kaggle/input contains: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'N/A'}")

print("Dataset Root located at:", dataset_root)
train_manifest_file = os.path.join(dataset_root, "train_manifest.jsonl")
val_manifest_file = os.path.join(dataset_root, "val_manifest.jsonl")

with open(train_manifest_file, "r", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]
with open(val_manifest_file, "r", encoding="utf-8") as f:
    val_records = [json.loads(line) for line in f]

print(f"Loaded {len(train_records)} training records and {len(val_records)} validation records.")

# PREFLIGHT VERIFICATION GATE
print("=" * 65)
print("PREFLIGHT DATASET INTEGRITY & DEMOGRAPHIC VERIFICATION")
print("=" * 65)
assert len(train_records) == 400, f"Expected 400 train records, got {len(train_records)}"
assert len(val_records) == 100, f"Expected 100 val records, got {len(val_records)}"

train_females = sum(1 for r in train_records if r["gender"] == "Female")
train_males = sum(1 for r in train_records if r["gender"] == "Male")
val_females = sum(1 for r in val_records if r["gender"] == "Female")
val_males = sum(1 for r in val_records if r["gender"] == "Male")

assert train_females == 200 and train_males == 200, f"Train gender mismatch: {train_females}F / {train_males}M"
assert val_females == 50 and val_males == 50, f"Val gender mismatch: {val_females}F / {val_males}M"

train_speakers = set(r["speaker_id"] for r in train_records)
val_speakers = set(r["speaker_id"] for r in val_records)
overlap = train_speakers & val_speakers
assert len(overlap) == 0, f"FATAL: Speaker leakage detected! Overlap: {overlap}"
assert len(train_speakers) == 40, f"Expected 40 train speakers, got {len(train_speakers)}"
assert len(val_speakers) == 20, f"Expected 20 val speakers, got {len(val_speakers)}"

print(f"  [PASS] Training Dataset: 400 records across 40 exclusive speakers ({train_females} Female, {train_males} Male)")
print(f"  [PASS] Validation Benchmark: 100 records across 20 benchmark speakers ({val_females} Female, {val_males} Male)")
print(f"  [PASS] Zero Acoustic Leakage: 0 overlapping speakers ({overlap})")

# Verify audio files exist on disk
missing_audio = 0
for rec in train_records + val_records:
    rel_path = rec["audio_filepath"]
    audio_path = os.path.join(dataset_root, rel_path)
    if not os.path.exists(audio_path):
        fname = os.path.basename(rel_path)
        candidates = glob.glob(os.path.join(dataset_root, "**", fname), recursive=True)
        if not candidates:
            missing_audio += 1
assert missing_audio == 0, f"FATAL: {missing_audio} audio files missing on disk!"
print(f"  [PASS] Audio Storage: All 500 audio files verified readable on disk!")
print("=" * 65)
print("TRAINING SETUP MATH & DYNAMICS:")
print("  Total Training Utterances    : 400")
print("  Per-Device Batch Size        : 1")
print("  Gradient Accumulation Steps  : 4 (Effective Batch Size = 4)")
print("  Total Optimization Steps     : 200")
print("  -> 200 steps x 4 accum = 800 utterance-level training passes")
print("  -> 800 passes / 400 utterances = EXACTLY 2.0 EPOCHS")
print("=" * 65)

# Helper to load and preprocess a single sample
def prepare_sample(record, base_dir, fe, tokenizer, device="cuda"):
    # Audio path
    rel_path = record["audio_filepath"]
    audio_path = os.path.join(base_dir, rel_path)
    if not os.path.exists(audio_path):
        fname = os.path.basename(rel_path)
        candidates = glob.glob(os.path.join(base_dir, "**", fname), recursive=True)
        if candidates:
            audio_path = candidates[0]
        else:
            audio_path = os.path.join(base_dir, fname)

    wav, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    wav = torch.from_numpy(wav.mean(axis=1))
    if sr != fe.sample_rate:
        wav = torchaudio.functional.resample(wav, sr, fe.sample_rate)

    batch = wav.unsqueeze(0).to(device)
    lens = torch.tensor([wav.shape[0]], device=device)
    features, feature_lens = fe(batch, lens)

    # Prompt + Target tokens
    prompt_ids = tokenizer.encode_prompt(lang="mr", itn=False, romanized=False)
    raw_pieces = tokenizer.multi.encode(record["text"])
    text_token_ids = [p + tokenizer.spl_size for p in raw_pieces]
    full_seq = prompt_ids + text_token_ids + [tokenizer.eos_id]

    decoder_input_ids = torch.tensor([full_seq[:-1]], dtype=torch.long, device=device)
    target_seq = full_seq[1:]
    labels_list = [-100 if idx < len(prompt_ids) - 1 else tok for idx, tok in enumerate(target_seq)]
    labels = torch.tensor([labels_list], dtype=torch.long, device=device)

    return features, decoder_input_ids, labels


In [ ]:
# 6. Multi-Step Fine-Tuning Execution
import math
import time
import torch.nn.functional as F

TOTAL_STEPS = 200
GRAD_ACCUM_STEPS = 4
BASE_LR = 1e-4
MIN_LR = 1e-5
WARMUP_STEPS = 20
LOG_INTERVAL = 10
CHECKPOINT_INTERVAL = 50

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=BASE_LR, betas=(0.9, 0.98), eps=1e-8, weight_decay=1e-4
)

def get_lr(step):
    if step < WARMUP_STEPS:
        return BASE_LR * (step + 1) / WARMUP_STEPS
    # Cosine decay
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress))

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
training_logs = []
step = 0
sample_idx = 0
num_train = len(train_records)

print("=" * 70)
print(f"STARTING FINE-TUNING RUN: {TOTAL_STEPS} Steps | Effective Batch Size: {GRAD_ACCUM_STEPS}")
print("=" * 70)

optimizer.zero_grad()
accum_loss = 0.0
t_start = time.time()

while step < TOTAL_STEPS:
    # Set LR
    current_lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = current_lr

    # Forward pass on 1 sample
    rec = train_records[sample_idx % num_train]
    sample_idx += 1

    features, dec_ids, labels = prepare_sample(rec, dataset_root, fe, tokenizer, device=device)

    enc_outputs = model.model.model.encoder(features, attention_mask=None)
    cross_mask = model.model._cross_mask_from_lengths(enc_outputs[1], enc_outputs[0].size(1))
    hidden = model.model.model.decoder(dec_ids, enc_outputs[0], cross_mask)
    logits = model.model.lm_head(hidden)

    loss = F.cross_entropy(logits.view(-1, tokenizer.vocab_size), labels.view(-1), ignore_index=-100, label_smoothing=0.1)
    loss_scaled = loss / GRAD_ACCUM_STEPS
    loss_scaled.backward()
    accum_loss += loss.item()

    if (sample_idx % GRAD_ACCUM_STEPS) == 0:
        # Gradient clipping
        grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_norm=1.0).item()
        optimizer.step()
        optimizer.zero_grad()
        step += 1

        avg_loss = accum_loss / GRAD_ACCUM_STEPS
        accum_loss = 0.0

        if step % LOG_INTERVAL == 0 or step == 1:
            elapsed = time.time() - t_start
            vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
            print(f"Step [{step:03d}/{TOTAL_STEPS}] | Loss: {avg_loss:.4f} | Grad Norm: {grad_norm:.4f} | LR: {current_lr:.2e} | VRAM: {vram_mb:.1f}MB | Elapsed: {elapsed:.1f}s")

            training_logs.append({
                "step": step,
                "loss": round(avg_loss, 4),
                "grad_norm": round(grad_norm, 4),
                "lr": current_lr,
                "vram_mb": round(vram_mb, 1),
                "elapsed_s": round(elapsed, 2)
            })

        # Save checkpoint
        if step % CHECKPOINT_INTERVAL == 0 or step == TOTAL_STEPS:
            ckpt_path = f"/kaggle/working/checkpoints/checkpoint_step_{step}.pt"
            lora_state = {k: v.cpu() for k, v in model.state_dict().items() if "lora" in k}
            torch.save({"step": step, "loss": avg_loss, "state_dict": lora_state}, ckpt_path)
            print(f"  --> Checkpoint saved: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024**2):.2f} MB)")

# Save final checkpoint
final_ckpt_path = "/kaggle/working/checkpoints/final_lora_checkpoint.pt"
lora_state = {k: v.cpu() for k, v in model.state_dict().items() if "lora" in k}
torch.save({"step": TOTAL_STEPS, "loss": avg_loss, "state_dict": lora_state}, final_ckpt_path)
print()
print(f"Training Complete! Final checkpoint saved to: {final_ckpt_path}")

with open("/kaggle/working/training_curves.json", "w", encoding="utf-8") as f:
    json.dump(training_logs, f, indent=2)


In [ ]:
# 7. Comparative Evaluation: Base vs Fine-Tuned Checkpoint on 100 Held-Out Utterances
import jiwer
import re

print("=" * 70)
print(f"EVALUATING ON 100 HELD-OUT VALIDATION UTTERANCES (20 Benchmark Speakers)")
print("=" * 70)

# Marathi Normalization Helper
def normalize_marathi(text):
    for ch in ['।', '॥', '।', ',', '?', '!', '.', ':', ';', '"', "'", '-']:
        text = text.replace(ch, ' ')
    return ' '.join(text.split())

# Greedy decoding helper
def transcribe_sample(m, audio_path):
    wav, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    wav = torch.from_numpy(wav.mean(axis=1))
    if sr != fe.sample_rate:
        wav = torchaudio.functional.resample(wav, sr, fe.sample_rate)
    batch = wav.unsqueeze(0).to(device)
    lens = torch.tensor([wav.shape[0]], device=device)
    features, flens = fe(batch, lens)

    with torch.no_grad():
        actual_m = m.model if hasattr(m, "model") else m
        enc_out = actual_m.model.encoder(features, attention_mask=None)
        c_mask = actual_m._cross_mask_from_lengths(enc_out[1], enc_out[0].size(1))

        prompt_ids = tokenizer.encode_prompt(lang="mr", itn=False, romanized=False)
        curr_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)

        max_new_tokens = 60
        for _ in range(max_new_tokens):
            h = actual_m.model.decoder(curr_ids, enc_out[0], c_mask)
            logits = actual_m.lm_head(h)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(-1)
            curr_ids = torch.cat([curr_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_id:
                break

        out_tokens = curr_ids[0, len(prompt_ids):].tolist()
        pred_text = tokenizer.decode(out_tokens)
    return pred_text

# Run evaluation on validation samples
eval_results = []
print("Evaluating base model vs fine-tuned model...")

for i, rec in enumerate(val_records):
    audio_path = os.path.join(dataset_root, rec["audio_filepath"])
    if not os.path.exists(audio_path):
        fname = os.path.basename(rec["audio_filepath"])
        candidates = glob.glob(os.path.join(dataset_root, "**", fname), recursive=True)
        if candidates:
            audio_path = candidates[0]
        else:
            audio_path = os.path.join(dataset_root, fname)

    ref_text = rec["text"]
    speaker = rec["speaker_id"]
    gender = rec["gender"]

    # 1. Base model prediction (with LoRA disabled)
    with model.disable_adapter():
        pred_base = transcribe_sample(model, audio_path)

    # 2. Fine-tuned model prediction (with LoRA enabled)
    pred_ft = transcribe_sample(model, audio_path)

    # Compute metrics
    wer_base = jiwer.wer(ref_text, pred_base) if ref_text else 1.0
    wer_ft = jiwer.wer(ref_text, pred_ft) if ref_text else 1.0
    cer_base = jiwer.cer(ref_text, pred_base) if ref_text else 1.0
    cer_ft = jiwer.cer(ref_text, pred_ft) if ref_text else 1.0

    wer_base_norm = jiwer.wer(normalize_marathi(ref_text), normalize_marathi(pred_base))
    wer_ft_norm = jiwer.wer(normalize_marathi(ref_text), normalize_marathi(pred_ft))

    eval_results.append({
        "sample_id": i,
        "speaker_id": speaker,
        "gender": gender,
        "reference": ref_text,
        "pred_base": pred_base,
        "pred_finetuned": pred_ft,
        "wer_base": round(wer_base, 4),
        "wer_finetuned": round(wer_ft, 4),
        "wer_base_norm": round(wer_base_norm, 4),
        "wer_finetuned_norm": round(wer_ft_norm, 4),
        "cer_base": round(cer_base, 4),
        "cer_finetuned": round(cer_ft, 4)
    })

    if (i + 1) % 25 == 0:
        print(f"  Evaluated {i+1} / {len(val_records)} samples...")

# Aggregate Telemetry
mean_wer_base = sum(r["wer_base"] for r in eval_results) / len(eval_results)
mean_wer_ft = sum(r["wer_finetuned"] for r in eval_results) / len(eval_results)
mean_cer_base = sum(r["cer_base"] for r in eval_results) / len(eval_results)
mean_cer_ft = sum(r["cer_finetuned"] for r in eval_results) / len(eval_results)
mean_wer_base_norm = sum(r["wer_base_norm"] for r in eval_results) / len(eval_results)
mean_wer_ft_norm = sum(r["wer_finetuned_norm"] for r in eval_results) / len(eval_results)

# Gender breakdown
female_base = sum(r["wer_base"] for r in eval_results if r["gender"] == "Female") / 50
female_ft = sum(r["wer_finetuned"] for r in eval_results if r["gender"] == "Female") / 50
male_base = sum(r["wer_base"] for r in eval_results if r["gender"] == "Male") / 50
male_ft = sum(r["wer_finetuned"] for r in eval_results if r["gender"] == "Male") / 50

print()
print("=" * 75)
print("EVALUATION BENCHMARK SUMMARY (N = 100 UTTERANCES)")
print("=" * 75)
print(f"Raw WER            : Base = {mean_wer_base*100:.2f}%  |  Fine-Tuned = {mean_wer_ft*100:.2f}%  (Delta: {(mean_wer_ft - mean_wer_base)*100:+.2f}%)")
print(f"Normalized WER     : Base = {mean_wer_base_norm*100:.2f}%  |  Fine-Tuned = {mean_wer_ft_norm*100:.2f}%  (Delta: {(mean_wer_ft_norm - mean_wer_base_norm)*100:+.2f}%)")
print(f"Raw CER            : Base = {mean_cer_base*100:.2f}%  |  Fine-Tuned = {mean_cer_ft*100:.2f}%  (Delta: {(mean_cer_ft - mean_cer_base)*100:+.2f}%)")
print(f"Female Subset WER  : Base = {female_base*100:.2f}%  |  Fine-Tuned = {female_ft*100:.2f}%")
print(f"Male Subset WER    : Base = {male_base*100:.2f}%  |  Fine-Tuned = {male_ft*100:.2f}%")
print("=" * 75)

summary_data = {
    "num_samples": len(eval_results),
    "mean_wer_base": round(mean_wer_base, 4),
    "mean_wer_finetuned": round(mean_wer_ft, 4),
    "mean_wer_base_norm": round(mean_wer_base_norm, 4),
    "mean_wer_finetuned_norm": round(mean_wer_ft_norm, 4),
    "mean_cer_base": round(mean_cer_base, 4),
    "mean_cer_finetuned": round(mean_cer_ft, 4),
    "gender_breakdown": {
        "female_wer_base": round(female_base, 4),
        "female_wer_finetuned": round(female_ft, 4),
        "male_wer_base": round(male_base, 4),
        "male_wer_finetuned": round(male_ft, 4)
    },
    "detailed_results": eval_results
}

with open("/kaggle/working/final_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2, ensure_ascii=False)
